#  Business Insights

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("data.csv", encoding="cp1252")

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

df["LineValue"] = df["Quantity"] * df["UnitPrice"]

df["InvoiceMonth"] = (
    df["InvoiceDate"].dt.to_period("M").astype("string")
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (541909, 10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineValue,InvoiceMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12


## Identify important patterns
A pattern is something that repeatedly appears in the data or a noticeable difference between groups.

Insight 1: Which countries contribute the most transaction value?
Business question: Which countries contribute the largest calculated transaction-line value?

In [2]:
country_summary = (
    df.groupby("Country")
    .agg(
        RecordCount=("InvoiceNo", "size"),
        TotalQuantity=("Quantity", "sum"),
        TotalLineValue=("LineValue", "sum")
    )
    .sort_values("TotalLineValue", ascending=False)
)

display(country_summary.head(10))

,RecordCount,TotalQuantity,TotalLineValue
Country,,,
United Kingdom,495478,4263829,8187806.364
Netherlands,2371,200128,284661.540
EIRE,8196,142637,263276.820
Germany,9495,117448,221698.210
France,8557,110480,197403.900
Australia,1259,83653,137077.270
Switzerland,2002,30325,56385.350
Spain,2533,26824,54774.580
Belgium,2069,23152,40910.960


### Insight 2: Which products contribute the most value?
Business question: Are a small number of products responsible for a large share of the recorded transaction value?

In [3]:
product_summary = (
    df.groupby(["StockCode", "Description"], dropna=False)
    .agg(
        TotalQuantity=("Quantity", "sum"),
        TotalLineValue=("LineValue", "sum"),
        RecordCount=("InvoiceNo", "size")
    )
    .sort_values("TotalLineValue", ascending=False)
)

display(product_summary.head(10))

,,TotalQuantity,TotalLineValue,RecordCount
StockCode,Description,,,
DOT,DOTCOM POSTAGE,707,206245.48,709
22423,REGENCY CAKESTAND 3 TIER,13033,164762.19,2200
47566,PARTY BUNTING,18022,98302.98,1727
85123A,WHITE HANGING HEART T-LIGHT HOLDER,35025,97715.99,2302
85099B,JUMBO BAG RED RETROSPOT,47363,92356.03,2159
23084,RABBIT NIGHT LIGHT,30680,66756.59,1051
POST,POSTAGE,3003,66230.64,1252
22086,PAPER CHAIN KIT 50'S CHRISTMAS,18902,63791.94,1210
84879,ASSORTED COLOUR BIRD ORNAMENT,36381,58959.73,1501


## Identify trends over time
A trend is a change in a measure over a period.
Insight 3: How does the recorded transaction value change each month?

In [4]:
monthly_summary = (
    df.dropna(subset=["InvoiceDate"])
    .groupby("InvoiceMonth")
    .agg(
        TotalLineValue=("LineValue", "sum"),
        TotalQuantity=("Quantity", "sum"),
        RecordCount=("InvoiceNo", "size")
    )
    .sort_index()
)

display(monthly_summary)

,TotalLineValue,TotalQuantity,RecordCount
InvoiceMonth,,,
2010-12,748957.020,342228,42481
2011-01,560000.260,308966,35147
2011-02,498062.650,277989,27707
2011-03,683267.080,351872,36748
2011-04,493207.121,289098,29916
2011-05,723333.510,380391,37030
2011-06,691123.120,341623,36874
2011-07,681300.111,391116,39518
2011-08,682680.510,406199,35284


## Identify relationships
Relationships show how two variables change together. They can help explain what is associated with a business outcome, but they do not automatically prove cause and effect.

In [5]:
relationship_df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
][["Quantity", "UnitPrice"]].dropna()

print(
    "Pearson correlation:",
    relationship_df["Quantity"].corr(relationship_df["UnitPrice"])
)

print(
    "Spearman correlation:",
    relationship_df["Quantity"].corr(
        relationship_df["UnitPrice"],
        method="spearman"
    )
)

Pearson correlation: -0.0037725426616366906
Spearman correlation: -0.4032372816383771


## Identify anomalies and potential business problems
An anomaly is an observation that differs from the usual pattern. Some anomalies are data errors; others are legitimate business events.

In [7]:
negative_quantity = df[df["Quantity"] < 0]

cancellation_mask = (
    df["InvoiceNo"].astype(str).str.startswith("C")
)

print("Records with negative quantity:", len(negative_quantity))
print("Records with cancellation-style invoice numbers:", cancellation_mask.sum())

display(
    df.loc[
        (df["Quantity"] < 0) | cancellation_mask,
        ["InvoiceNo", "StockCode", "Quantity",
         "UnitPrice", "InvoiceDate", "Country"]
    ].head(10)
)

Records with negative quantity: 10624
Records with cancellation-style invoice numbers: 9288


,InvoiceNo,StockCode,Quantity,UnitPrice,InvoiceDate,Country
141,C536379,D,-1,27.50,2010-12-01 09:41:00,United Kingdom
154,C536383,35004C,-1,4.65,2010-12-01 09:49:00,United Kingdom
235,C536391,22556,-12,1.65,2010-12-01 10:24:00,United Kingdom
236,C536391,21984,-24,0.29,2010-12-01 10:24:00,United Kingdom
237,C536391,21983,-24,0.29,2010-12-01 10:24:00,United Kingdom
238,C536391,21980,-24,0.29,2010-12-01 10:24:00,United Kingdom
239,C536391,21484,-12,3.45,2010-12-01 10:24:00,United Kingdom
240,C536391,22557,-12,1.65,2010-12-01 10:24:00,United Kingdom
241,C536391,22553,-24,1.65,2010-12-01 10:24:00,United Kingdom
939,C536506,22960,-6,4.25,2010-12-01 12:38:00,United Kingdom


## Identify interesting business segments
A segment is a group of records that shares a meaningful characteristic, such as country, product, customer, or time period.

In [8]:
country_segments = (
    df.groupby("Country")
    .agg(
        RecordCount=("InvoiceNo", "size"),
        AverageLineValue=("LineValue", "mean"),
        TotalLineValue=("LineValue", "sum")
    )
)

# Focus on countries with at least 100 records
country_segments = country_segments[
    country_segments["RecordCount"] >= 100
]

display(
    country_segments
    .sort_values("AverageLineValue", ascending=False)
    .head(10)
)

,RecordCount,AverageLineValue,TotalLineValue
Country,,,
Netherlands,2371,120.059696,284661.54
Australia,1259,108.877895,137077.27
Japan,358,98.716816,35340.62
Sweden,462,79.211926,36595.91
Denmark,389,48.247147,18768.14
Singapore,229,39.827031,9120.39
Hong Kong,288,35.128611,10117.04
Norway,1086,32.378877,35163.46
Greece,146,32.263836,4710.52


## Identify potential business opportunities
Business opportunities are possible improvements suggested by the findings. They should be validated before decisions are made.

In [9]:
product_demand = (
    df[df["Quantity"] > 0]
    .groupby("StockCode")
    .agg(
        TotalPositiveQuantity=("Quantity", "sum"),
        RecordCount=("InvoiceNo", "size"),
        AverageUnitPrice=("UnitPrice", "mean")
    )
    .sort_values("TotalPositiveQuantity", ascending=False)
)

display(product_demand.head(10))

,TotalPositiveQuantity,RecordCount,AverageUnitPrice
StockCode,,,
23843,80995,1,2.080000
23166,78033,250,1.468480
22197,56921,1426,1.042468
84077,55047,537,0.320670
85099B,48478,2115,2.481934
85123A,41664,2270,3.110084
84879,36461,1489,1.722290
21212,36419,1370,0.758431
23084,31722,1049,2.351001
